# Day 3 · Your First Agent & LangChain

**Roll Number:** `24P31A1201`  
**Program:** Drive Ready – AI Agents | One Person Company (OPC) Initiative  
**Prerequisite:** Add `GROQ_API_KEY` in Google Colab **Secrets** (key icon on the left sidebar) and enable **Notebook access**.

In [ ]:
# Cell 1: Install LangChain + Groq integration
!pip install -q langchain langchain-groq

In [ ]:
# Cell 2: Load GROQ_API_KEY from Colab Secrets and initialize ChatGroq
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

MODEL = "openai/gpt-oss-120b"
llm = ChatGroq(model=MODEL, temperature=0.3)
print("ChatGroq ready with", MODEL)

In [ ]:
# Cell 3: Prompt template + memory + LCEL chain (prompt | llm | StrOutputParser)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes and cupcakes have eggless versions.
RULES:  Never promise a delivery time or place.
        If asked for same-day or today's order, clearly explain that orders need 24 hours' notice.
        Only say "I'll check with the owner." when the question is not covered in FACTS (like delivery areas).
        Always write "eggless" (not "egg-less"). Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

In [ ]:
%%writefile tests.csv
question,must_include
How much is a plum cake?,500
Do you make eggless cakes?,eggless
Can I get a cake today evening?,24 hours
Do you deliver to Kakinada?,check with the owner
What is the price of 6 cupcakes?,480
How much does a chocolate cake cost?,650
What is the total for 2 plum cakes?,1000
Are cupcakes available?,80
Can you make an eggless chocolate cake?,eggless
Do you offer free home delivery?,check with the owner

In [ ]:
# Cell 5: Run automated tests (tests.csv)
import csv

def run_tests(path: str = "tests.csv", target_chain=None) -> float:
    active_chain = target_chain or chain
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    passed = 0
    for row in rows:
        answer = active_chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "->", answer[:80])
    rate = passed / len(rows)
    print(f"\n{passed}/{len(rows)} passed ({rate:.0%})")
    return rate

run_tests()

In [ ]:
# Cell 6: Temperature experiment (temperature=1.0 run twice)
llm_hot = ChatGroq(model=MODEL, temperature=1.0)
chain_hot = prompt | llm_hot | StrOutputParser()

print("=== Run 1 (temperature=1.0) ===")
run_tests(target_chain=chain_hot)

print("\n=== Run 2 (temperature=1.0) ===")
run_tests(target_chain=chain_hot)

In [ ]:
# Cell 7: Sneak peek — the same assistant as a LangChain agent
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)  # tools come on Day 4
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

In [ ]:
# Cell 8: Day 4 Prep — Two tool-ready Python functions with type hints and docstrings
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

def get_price(item: str, qty: int = 1) -> str:
    """Return the total price in rupees for qty of a bakery menu item."""
    key = item.lower().strip()
    if key not in MENU:
        return f"'{item}' is not on our menu. Available items: {', '.join(MENU)}."
    return f"Rs {MENU[key] * qty} for {qty} {key}(s)."

def check_availability(item: str, hours_notice: int = 24) -> str:
    """Check whether an item is on the menu and meets the 24-hour advance notice rule."""
    key = item.lower().strip()
    if key not in MENU:
        return f"'{item}' is not on the menu. I'll check with the owner."
    if hours_notice < 24:
        return f"Orders need at least 24 hours' notice, so {hours_notice} hours is too short."
    return f"Yes, {key} (including eggless) is available with {hours_notice} hours' notice."

print(get_price("cupcake", 6))
print(check_availability("plum cake", 12))
print(check_availability("plum cake", 36))